# レッスン 11 - モデルコンテキストプロトコル (MCP)

The **モデルコンテキストプロトコル (MCP)** は、エージェントが実行時にツール、リソース、データソースを動的に発見して使用できるようにするオープンな標準です。ツールをエージェントにハードコーディングする代わりに、MCPはエージェントが要求に応じて機能を公開する外部サーバーに接続できるようにします。

このレッスンでは、次のことを学びます:
- MCPとは何か、そしてエージェントシステムにとってなぜ重要か
- MCPのクライアント・サーバーアーキテクチャがどのように機能するか
- MCPスタイルのツール検出を使用するエージェントの構築方法


## セットアップ

**前提条件:**
- Azure AI Foundry プロジェクト（デプロイされたモデルを含む）
- 認証のために `az login` を実行


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Model Context Protocol (MCP)とは何ですか？

MCPは、AIエージェントが外部ツールやデータソースを発見し、相互作用するための標準的な方法を定義します：

- **MCP Server**: 標準プロトコルを通じてツール、リソース、プロンプトを公開します
- **MCP Client**: サーバーに接続し、利用可能な機能を検出するエージェント実行環境
- **Dynamic Discovery**: エージェントはハードコードされたツールを必要とせず — 実行時に利用可能なものを発見します

これは、エージェントのコードを変更せずに新しい機能を追加できる、拡張可能なエージェントシステムを構築する際に強力です。


## MCP の仕組み

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. エージェント (MCP クライアント) が MCP サーバーに接続する
2. サーバーは利用可能なツールの一覧とそれらのスキーマを返す
3. エージェントは推論の過程で発見した任意のツールを呼び出すことができる
4. 結果は同じプロトコルを通じて戻ってくる


## MCPツールの検出をシミュレート

実際のMCPサーバーは実行中のサーバープロセスを必要とするため、ここでは`@tool`関数を使用して、MCPに接続された宿泊施設サービスが提供するものをシミュレートするパターンを示します。

本番環境では、これらのツールはローカルで定義されるのではなく、MCPサーバーから動的に発見されます。


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## MCPスタイルのツールを使ったエージェントの構築


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## 本番環境での MCP

本番環境では、MCP は強力なパターンを可能にします:

- **動的なツール検出**: エージェントは MCP サーバーに接続し、実行時にツールを検出します
- **疎結合アーキテクチャ**: ツール提供者はエージェントとは独立して更新できます
- **組織間共有**: チームは MCP サーバーを介して機能を公開し、任意のエージェントが使用できるようにできます
- **Microsoft Agent Framework のサポート**: MAF は `mcp` 統合を通じて組み込みの MCP クライアントサポートを提供します

MAF で実際の MCP サーバーを使用するには、`hosted_mcp_tool()` または MCP クライアント統合を介して接続します。

**詳細:**
- [MCP Specification](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP Support](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## まとめ

このレッスンで学んだこと:
- **MCP** はエージェントとツール提供者の間で動的にツールを検出するためのオープン標準です
- **クライアントサーバーアーキテクチャ** により、エージェントは実行時に機能を検出できます
- MCP は **拡張可能で疎結合なエージェントシステム** を可能にし、コード変更なしでツールを追加できます
- Microsoft Agent Framework は本番環境向けに **組み込みの MCP サポート** を提供します


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
免責事項：
本書は AI 翻訳サービス「Co-op Translator」(https://github.com/Azure/co-op-translator) を用いて翻訳されました。正確性の確保に努めていますが、自動翻訳には誤りや不正確な表現が含まれる可能性があります。原文（原言語）の文書を権威ある情報源とみなしてください。重要な情報については、専門の人による翻訳を推奨します。本翻訳の利用に伴う誤解や誤訳について、当方は一切責任を負いません。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
